# 🔍 Introduction to Spark SQL in Python — Project Showcase

![PySpark](https://img.shields.io/badge/Library-PySpark-E25A1C?logo=apachespark&logoColor=white)
![Spark SQL](https://img.shields.io/badge/Module-Spark%20SQL-blueviolet)
![Spark ML](https://img.shields.io/badge/ML-Spark%20MLlib-00ADD8)
![Platform](https://img.shields.io/badge/Platform-DataCamp-05192D?logo=datacamp&logoColor=white)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

> **Platform:** DataCamp | **Track:** Associate Data Engineer in Databricks (Course 12)
> **Instructor:** Mark Plutowski

**Datasets used in the exercises:** a Caltrain-style **train schedule** (`trainsched.txt`: `train_id`, `station`, `time`) and the full text of **The Adventures of Sherlock Holmes** (Project Gutenberg), processed into clauses, words and chapters.

**Objectives**
1. Query DataFrames with SQL, and express the same logic with window functions in both SQL and dot notation.
2. Use window functions for natural language processing: tokenizing, moving windows and word-sequence (n-gram) counts.
3. Make Spark jobs efficient and observable: caching, the Spark UI, logging without hidden actions, and reading query plans.
4. Build an end-to-end text classifier: UDFs, feature vectors, labeling, train/test split, logistic regression and evaluation.

The exercises ran in DataCamp's Spark environment (with a preloaded `spark` session and data), so code cells are transcriptions of my solutions and are not meant to be re-run here. Results are shown with cropped screenshots of the console (`./screenshots/`).

| Chapter | Topic |
|---|---|
| 1 | PySpark SQL and window functions |
| 2 | Window function SQL for natural language processing |
| 3 | Caching, logging and the Spark UI |
| 4 | Text classification |

## Chapter 1 — PySpark SQL

### 1.1 — Creating and querying a SQL table

A DataFrame becomes a SQL table by registering it as a **temporary view**: it exists only for the session, and `createOrReplaceTempView` creates it or replaces it with the current data.

In [ ]:
df = spark.read.csv("trainsched.txt", header=True)
df.createOrReplaceTempView("schedule")
spark.sql("SELECT * FROM schedule WHERE station = 'San Jose'").show()

# Three ways to inspect a table's columns when there's no documentation
spark.sql("SHOW COLUMNS FROM schedule").show()
spark.sql("SELECT * FROM schedule LIMIT 0").columns
spark.sql("DESCRIBE schedule").show()

**Exercise — creating a SQL table from a DataFrame.**

In [ ]:
# Load trainsched.txt
df = spark.read.csv("trainsched.txt", header=True)

# Create temporary table called table1
df.createOrReplaceTempView('table1')

**Exercise — determining the column names of a table.** In practice, tables often come without a documented schema, so `DESCRIBE` is the first query I run.

In [ ]:
# Inspect the columns in the table schedule
spark.sql("DESCRIBE schedule").show()

![All three columns of schedule are strings](./screenshots/ch1_ex2_describe_schedule.png)
*All three columns of schedule are strings*

**Interpretation:** `train_id`, `station` and `time` are all `string` — the CSV was read without type inference, so `time` (e.g. `6:06a`) would need conversion before any time arithmetic.

### 1.2 — Window function SQL

A **window function** is like an aggregate, but instead of collapsing each group into one row, it returns a value **for every row**, computed from other rows in its window. This makes "compare with the next/previous row" or "running total" queries much simpler (and faster) than self-joins.

- `OVER (ORDER BY time)` defines the order of rows in the window.
- `PARTITION BY train_id` restarts the window for each group, so values never leak across groups.
- `LEAD(col, n)` reads the value `n` rows ahead; `LAG(col, n)` reads `n` rows behind; `ROW_NUMBER()` numbers rows within the window.

In [ ]:
query = """
SELECT train_id, station, time,
       LEAD(time, 1) OVER (PARTITION BY train_id ORDER BY time) AS time_next
FROM schedule
"""
spark.sql(query).show()

**Exercise — running sums with a window function.** `diff_min` is the minutes from each station to the next one; a running total accumulates it along each line.

In [ ]:
# Add col running_total that sums diff_min col in each group
query = """
SELECT train_id, station, time, diff_min,
SUM(diff_min) OVER (PARTITION BY train_id ORDER BY time) AS running_total
FROM schedule
"""

# Run the query and display the result
spark.sql(query).show()

![Cumulative travel minutes per train, restarting for each train_id](./screenshots/ch1_ex3_running_total.png)
*Cumulative travel minutes per train, restarting for each train_id*

**Interpretation:** train 217 reaches 53 minutes by Tamien; at the last stop (San Jose) `diff_min` is null and the running total stays at 53, because `SUM` ignores nulls.

**Exercise — fixing a broken query.** This query runs, but one row is wrong:

In [ ]:
query = """
SELECT
ROW_NUMBER() OVER (ORDER BY time) AS row,
train_id,
station,
time,
LEAD(time,1) OVER (ORDER BY time) AS time_next
FROM schedule
"""
spark.sql(query).show()

# Give the number of the bad row as an integer
bad_row = 7

# Provide the missing clause, SQL keywords in upper case
clause = 'PARTITION BY train_id'

![Without PARTITION BY, row 7 (San Jose, train 217) takes its time_next from train 324](./screenshots/ch1_ex4_buggy_lead_row7.png)
*Without PARTITION BY, row 7 (San Jose, train 217) takes its time_next from train 324*

**Interpretation:** row 7 is the last stop of train 217, but its `time_next` is `7:59a` — the first stop of a **different train**. Adding `PARTITION BY train_id` to the `OVER` clause keeps each window inside one train, so that row correctly becomes null.

### 1.3 — Dot notation and SQL

Every SQL operation has a DataFrame ("dot notation") equivalent. Choosing one is mostly preference, but some things are clearly simpler in one style.

In [ ]:
from pyspark.sql.functions import col

# Three ways to select two columns
df.select('train_id', 'station')
df.select(df.train_id, df.station)
df.select(col('train_id'), col('station'))

# Two ways to rename a column
df.select('train_id', 'station').withColumnRenamed('train_id', 'train')
df.select(col('train_id').alias('train'), 'station')

# Don't mix styles in one call — it works, but hurts readability
df.select('train_id', df.station, col('time'))

**Window functions in dot notation:** `ROW_NUMBER` → `pyspark.sql.functions.row_number`, the inside of `OVER` → `pyspark.sql.Window`, `PARTITION BY` → `Window.partitionBy`, `ORDER BY` → `Window.orderBy`. A window definition is a reusable `WindowSpec` object.

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, lead

df.withColumn("id", row_number().over(Window.partitionBy('train_id').orderBy('time')))

window = Window.partitionBy('train_id').orderBy('time')   # type: WindowSpec
dfx = df.withColumn('next', lead('time', 1).over(window))

**Exercise — aggregation, step by step.** The first pair of commands must return identical results; the second pair shows a trap.

In [ ]:
# Give the identical result in each command
spark.sql('SELECT train_id, MIN(time) AS start FROM schedule GROUP BY train_id').show()
df.groupBy('train_id').agg({'time':'MIN'}).withColumnRenamed('MIN(time)', 'start').show()

# Print the second column of the result
spark.sql('SELECT train_id, MIN(time), MAX(time) FROM schedule GROUP BY train_id').show()
result = df.groupBy('train_id').agg({'time':'min', 'time':'max'})
result.show()
print(result.columns[1])

![SQL and dot notation give the same first departure per train](./screenshots/ch1_ex5_min_time_two_ways.png)
*SQL and dot notation give the same first departure per train*

![The dictionary version returns only max(time): the second aggregation overwrote the first](./screenshots/ch1_ex5_agg_dict_clobber.png)
*The dictionary version returns only max(time): the second aggregation overwrote the first*

**Interpretation:** `{'time':'min', 'time':'max'}` is a Python dictionary with a **duplicate key**, so only the last entry survives — the `min` silently disappears and `result.columns[1]` is `max(time)`. The dictionary form of `.agg()` can do only one aggregation per column.

**Exercise — aggregating the same column twice.** The dot-notation fix uses functions with aliases:

In [ ]:
from pyspark.sql.functions import min, max, col

expr = [min(col("time")).alias('start'), max(col("time")).alias('end')]
dot_df = df.groupBy("train_id").agg(*expr)
dot_df.show()

My task was to produce the same result in SQL, where it's much simpler:

In [ ]:
# Write a SQL query giving a result identical to dot_df
query = """
        SELECT train_id,
        MIN(time) AS start,
        MAX(time) AS end
        FROM schedule
        GROUP BY train_id
        """
sql_df = spark.sql(query)
sql_df.show()

![First and last time per train computed in SQL](./screenshots/ch1_ex6_first_last_time_sql.png)
*First and last time per train computed in SQL*

**Exercise — converting SQL to dot notation.** Reproduce this SQL with the `Window` API:

```sql
SELECT *,
       LEAD(time, 1) OVER (PARTITION BY train_id ORDER BY time) AS time_next
FROM schedule
```

In [ ]:
# Obtain the identical result using dot notation
from pyspark.sql import Window
from pyspark.sql.functions import lead

dot_df = df.withColumn('time_next', lead('time', 1)
        .over(Window.partitionBy('train_id')
        .orderBy('time')))

dot_df.show()

![time_next per train computed with lead().over(Window)](./screenshots/ch1_ex7_lead_dot_notation.png)
*time_next per train computed with lead().over(Window)*

**Exercise — converting a window function from dot notation to SQL.** The dot-notation version computes minutes to the next stop with `unix_timestamp` (equivalent to the SQL `UNIX_TIMESTAMP` function):

```python
window = Window.partitionBy('train_id').orderBy('time')
dot_df = df.withColumn('diff_min',
                       (unix_timestamp(lead('time', 1).over(window), 'H:m')
                        - unix_timestamp('time', 'H:m')) / 60)
```

In [ ]:
# Create a SQL query to obtain an identical result to dot_df
query = """
SELECT *,
(UNIX_TIMESTAMP(LEAD(time, 1) OVER (PARTITION BY train_id ORDER BY time), 'H:m')
 - UNIX_TIMESTAMP(time, 'H:m'))/60 AS diff_min
FROM schedule
"""
sql_df = spark.sql(query)
sql_df.show()

![Minutes to the next stop for each station](./screenshots/ch1_ex8_diff_min_sql.png)
*Minutes to the next stop for each station*

**Interpretation:** the timestamps are converted to seconds, subtracted and divided by 60. These are the same `diff_min` values used for the running total in 1.2, and the last stop is null because it has no next stop.

## Chapter 2 — Window Function SQL for Natural Language Processing

### 2.1 — Loading and preparing text

The dataset is *The Adventures of Sherlock Holmes*. Text can be read line by line, or from a Parquet file that already holds processed data:

In [ ]:
from pyspark.sql.functions import lower, col, regexp_replace, split, explode, length, monotonically_increasing_id, when

df = spark.read.text('sherlock.txt')        # one row per line, column "value"
df1 = spark.read.load('sherlock.parquet')   # load() defaults to Parquet

df = df1.select(lower(col('value')).alias('v'))                         # lower case + clean name
df = df1.select(regexp_replace('value', r'Mr\.', 'Mr').alias('v'))      # "Mr. Holmes." -> "Mr Holmes."
df = df1.select(regexp_replace('value', "don't", 'do not').alias('v'))  # expand contractions

punctuation = "_|.\\?\\!\",\\'\\[\\]\\*()"
df3 = df2.select(split('v', '[ %s]' % punctuation).alias('words'))      # tokenize; split characters are discarded
df4 = df3.select(explode('words').alias('word'))                        # one row per word
nonblank_df = df4.where(length('word') > 0)                             # drop empty tokens

df2 = nonblank_df.select('word', monotonically_increasing_id().alias('id'))  # word position

`explode` turns an array into one row per element, so the row count grows; splitting on punctuation leaves empty strings, which `length(...) > 0` removes. The `id` keeps the original word order, which window functions need later.

**Exercise — loading a DataFrame from a Parquet file.**

In [ ]:
# Load the dataframe
df = spark.read.load('sherlock_sentences.parquet')

# Filter and show the first 5 rows
df.where('id > 70').show(5, truncate=False)

![Clauses with id above 70](./screenshots/ch2_ex1_parquet_clauses.png)
*Clauses with id above 70*

**Exercise — splitting and exploding a text column.**

In [ ]:
# Split the clause column into a column called words
split_df = clauses_df.select(split('clause', ' ').alias('words'))
split_df.show(5, truncate=False)

# Explode the words column into a column called word
exploded_df = split_df.select(explode('words').alias('word'))
exploded_df.show(10)

# Count the resulting number of rows in exploded_df
print("\nNumber of rows: ", exploded_df.count())

![Each clause split into an array of words](./screenshots/ch2_ex2_split_words.png)
*Each clause split into an array of words*

![After explode, each word is its own row](./screenshots/ch2_ex2_explode_words.png)
*After explode, each word is its own row*

**Interpretation:** the 100 clauses become one row per word — the shape needed for counting and windowing.

### 2.2 — Partitioning the text

The text can be labelled by section with `when()` and physically **repartitioned** on that column, so each chapter's rows stay on one machine:

In [ ]:
df2 = df.withColumn('title', when(df.id < 25000, 'Preface')
                             .when(df.id < 50000, 'Chapter 1')
                             .when(df.id < 75000, 'Chapter 2')
                             .otherwise('Chapter 3'))

df2 = df.repartition(4, 'part')
print(df2.rdd.getNumPartitions())

df_parts = spark.read.text('sherlock_parts')   # a folder of files is read as one DataFrame

### 2.3 — Moving window analysis

With words indexed by `id` and grouped by `part` (chapter), `LEAD` and `LAG` build a **moving window** of neighbouring words — the context-window features many NLP models use. `PARTITION BY part` keeps windows inside a chapter: the first words of a chapter get nulls instead of words from the previous one.

In [ ]:
query = """
SELECT id, word AS w1,
       LEAD(word, 1) OVER (PARTITION BY part ORDER BY id) AS w2,
       LEAD(word, 2) OVER (PARTITION BY part ORDER BY id) AS w3
FROM df
"""
spark.sql(query).sort('id').show()

**Exercise — creating context window feature data.** For each word (`w3`), get the two previous and two following words within the same chapter.

In [ ]:
# Word for each row, previous two and subsequent two words
query = """
SELECT
part,
LAG(word, 2) OVER(PARTITION BY part ORDER BY id) AS w1,
LAG(word, 1) OVER(PARTITION BY part ORDER BY id) AS w2,
word AS w3,
LEAD(word, 1) OVER(PARTITION BY part ORDER BY id) AS w4,
LEAD(word, 2) OVER(PARTITION BY part ORDER BY id) AS w5
FROM text
"""
spark.sql(query).where("part = 12").show(10)

![A 5-word sliding window over chapter 12](./screenshots/ch2_ex3_context_window.png)
*A 5-word sliding window over chapter 12*

**Interpretation:** the first word of chapter 12 (`xii`) has null `w1` and `w2` — there are no earlier words *within that chapter*, which is exactly what the partition guarantees.

**Exercise — repartitioning the data.** The next steps group by chapter, so I repartitioned the single-partition `text_df` into one partition per chapter to avoid shuffling data between machines.

In [ ]:
# Repartition text_df into 12 partitions on 'chapter' column
repart_df = text_df.repartition(12, 'chapter')

# Prove that repart_df has 12 partitions
print(repart_df.rdd.getNumPartitions())

**Result:** `12`.

### 2.4 — Common word sequences

Words are **categorical** data (`he`, `she`, `that`), not ordinal, so the natural analysis is counting. Using the moving-window query as a **subquery**, I can count how often each sequence of words (an n-gram) appears:

In [ ]:
query3agg = """
SELECT w1, w2, w3, COUNT(*) AS count FROM (
    SELECT word AS w1,
           LEAD(word, 1) OVER (PARTITION BY part ORDER BY id) AS w2,
           LEAD(word, 2) OVER (PARTITION BY part ORDER BY id) AS w3
    FROM df
)
GROUP BY w1, w2, w3
ORDER BY count DESC
"""
spark.sql(query3agg).show()

In the whole book the most common 3-word sequence is *"one of the"* (49 times), followed by *"i think that"* and *"it is a"*. The same subquery can also rank sequences by other measures, such as total length of the words.

**Exercise — finding common word sequences (5-tuples).** The `text` table holds the first five chapters (columns `word`, `id`, `part`, `title`):

![Structure of the text table: one word per row with its position, chapter and title](./screenshots/ch2_ex5_text_table_preview.png)
*Structure of the text table: one word per row with its position, chapter and title*

In [ ]:
# Find the top 10 sequences of five words
query = """
SELECT w1, w2, w3, w4, w5, COUNT(*) AS count FROM (
   SELECT word AS w1,
   LEAD(word, 1) OVER(PARTITION BY part ORDER BY id) AS w2,
   LEAD(word, 2) OVER(PARTITION BY part ORDER BY id) AS w3,
   LEAD(word, 3) OVER(PARTITION BY part ORDER BY id) AS w4,
   LEAD(word, 4) OVER(PARTITION BY part ORDER BY id) AS w5
   FROM text
)
GROUP BY w1, w2, w3, w4, w5
ORDER BY count DESC, w1, w2, w3, w4, w5
LIMIT 10;
"""
df = spark.sql(query)
df.show()

![The ten most frequent 5-word sequences](./screenshots/ch2_ex5_top_5_tuples.png)
*The ten most frequent 5-word sequences*

**Interpretation:** *"in the case of the"* is the only 5-tuple seen 4 times; many others tie at 3. That's why the `ORDER BY` continues with `w1 … w5` after `count DESC`: without a **tie-breaker**, rows with the same count can come back in any order, and the result would not be reproducible between runs.

**Exercise — unique 5-tuples in sorted order.** The last ten unique 5-tuples, sorted alphabetically in descending order:

In [ ]:
# Unique 5-tuples sorted in descending order
query = """
SELECT DISTINCT w1, w2, w3, w4, w5 FROM (
   SELECT word AS w1,
   LEAD(word,1) OVER(PARTITION BY part ORDER BY id ) AS w2,
   LEAD(word,2) OVER(PARTITION BY part ORDER BY id ) AS w3,
   LEAD(word,3) OVER(PARTITION BY part ORDER BY id ) AS w4,
   LEAD(word,4) OVER(PARTITION BY part ORDER BY id ) AS w5
   FROM text
)
ORDER BY w1 DESC, w2 DESC, w3 DESC, w4 DESC, w5 DESC
LIMIT 10
"""
df = spark.sql(query)
df.show()

![Unique 5-tuples in reverse alphabetical order](./screenshots/ch2_ex6_unique_5_tuples_desc.png)
*Unique 5-tuples in reverse alphabetical order*

**Interpretation:** `DISTINCT` applies to the **whole combination** of selected columns, not just `w1` — that's why `youth` appears four times with different continuations. A row is removed only if all five words match another row.

**Exercise — most frequent 3-tuple per chapter.** The provided `subquery` counts 3-tuples per chapter; I wrapped it in a query that keeps the top one per chapter with `ROW_NUMBER()`:

In [ ]:
subquery = """
SELECT chapter, w1, w2, w3, COUNT(*) as count
FROM
(
    SELECT
    chapter,
    word AS w1,
    LEAD(word, 1) OVER(PARTITION BY chapter ORDER BY id ) AS w2,
    LEAD(word, 2) OVER(PARTITION BY chapter ORDER BY id ) AS w3
    FROM text
)
GROUP BY chapter, w1, w2, w3
ORDER BY chapter, count DESC
"""

#   Most frequent 3-tuple per chapter
query = """
SELECT chapter, w1, w2, w3, count FROM
(
  SELECT
  chapter,
  ROW_NUMBER() OVER (PARTITION BY chapter ORDER BY count DESC) AS row,
  w1, w2, w3, count
  FROM ( %s )
)
WHERE row = 1
ORDER BY chapter ASC
""" % subquery

spark.sql(query).show()

![The most frequent 3-word sequence in each of the 12 chapters](./screenshots/ch2_ex7_top_3_tuple_per_chapter.png)
*The most frequent 3-word sequence in each of the 12 chapters*

**Interpretation:** the top sequences reveal each story's main characters — *"mr hosmer angel"* (chapter 3), *"neville st clair"* (6), *"dr grimesby roylott"* (8), *"lord st simon"* (10, 28 times) — showing how simple n-gram counts already surface named entities.

The `%s` / `% subquery` part is plain Python string formatting: `%s` is a placeholder in the query text, and `% subquery` inserts the earlier query there, so it can be reused as a subquery without copying it.

## Chapter 3 — Caching, Logging and the Spark UI

### 3.1 — Caching

Caching keeps data in memory for reuse. Spark unloads memory aggressively using a **Least Recently Used (LRU)** eviction policy, independently on each worker and depending on the memory available, so caching is a request, not a guarantee.

In [ ]:
import pyspark

df.cache()          # same as df.persist() -> MEMORY_AND_DISK in Spark 2.1+
df.is_cached        # True
df.storageLevel     # StorageLevel(True, True, False, True, 1): useDisk, useMemory, useOffHeap, deserialized, replication
df.unpersist()

df.persist(storageLevel=pyspark.StorageLevel.MEMORY_AND_DISK)

# Tables are cached through the catalog
df.createOrReplaceTempView('df')
spark.catalog.cacheTable('df')
spark.catalog.isCached(tableName='df')
spark.catalog.uncacheTable('df')
spark.catalog.clearCache()

**Tips:** caching is lazy (nothing is stored until an action runs); cache only data used by more than one operation; cache selectively; and unpersist when done.

**Exercise — caching `df1` only.** `df2` is derived from `df1` with extra compute-intensive steps. Each action is timed:

In [ ]:
# Unpersists df1 and df2 and initializes a timer
prep(df1, df2)

# Cache df1
df1.cache()

# Run actions on both dataframes
run(df1, "df1_1st")
run(df1, "df1_2nd")
run(df2, "df2_1st")
run(df2, "df2_2nd", elapsed=True)

# Prove df1 is cached
print(df1.is_cached)

![Timings with df1 cached](./screenshots/ch3_ex1_cache_df1_timings.png)
*Timings with df1 cached*

**Exercise — caching `df2` only.**

In [ ]:
# Unpersist df1 and df2 and initializes a timer
prep(df1, df2)

# Persist df2 using memory and disk storage level
df2.persist(storageLevel=pyspark.StorageLevel.MEMORY_AND_DISK)

# Run actions both dataframes
run(df1, "df1_1st")
run(df1, "df1_2nd")
run(df2, "df2_1st")
run(df2, "df2_2nd", elapsed=True)

![Timings with df2 persisted](./screenshots/ch3_ex2_persist_df2_timings.png)
*Timings with df2 persisted*

**Comparison:**

| Action | Cache `df1` | Persist `df2` |
|---|---|---|
| df1 1st | 4.0 s | 3.4 s |
| df1 2nd | **0.1 s** | 1.8 s |
| df2 1st | **0.2 s** | 2.1 s |
| df2 2nd | 0.1 s | **0.1 s** |
| **Total** | **4.9 s** | 7.8 s |

**Interpretation:** caching the upstream `df1` paid off for *both* DataFrames, because `df2` is built from `df1` and reused its cached data. Caching only `df2` helped only repeated actions on `df2`, while `df1` was recomputed each time. Here, caching the shared parent was the better choice.

**Exercise — caching and uncaching tables.**

In [ ]:
# List the tables
print("Tables:\n", spark.catalog.listTables())

# Cache table1 and confirm that it is cached
spark.catalog.cacheTable('table1')
print("table1 is cached: ", spark.catalog.isCached('table1'))

# Uncache table1 and confirm that it is uncached
spark.catalog.uncacheTable('table1')
print("table1 is cached: ", spark.catalog.isCached('table1'))

![table1 is a temporary table; cached: True, then False](./screenshots/ch3_ex3_cache_uncache_table.png)
*table1 is a temporary table; cached: True, then False*

### 3.2 — The Spark UI

The Spark UI (at `http://[DRIVER_HOST]:4040`, or 4041, 4042… if that port is taken) shows how work is executed:
- **Task:** a unit of execution on a single CPU.
- **Stage:** a group of tasks doing the same computation in parallel, each on a different subset of the data.
- **Job:** a computation triggered by an action, split into one or more stages.

The **Storage** tab shows where data partitions are cached (memory or disk, across the cluster) at a point in time; the **SQL** tab shows the queries that ran.

**Spark UI questions.**
- A folder of 12 text files is loaded into a DataFrame with **7 partitions**, registered as table `text` and cached with `spark.catalog.cacheTable('text')`. Once an action triggers the cache, the Storage tab shows an RDD named **"In-memory table text"** with storage level *Memory Deserialized 1x Replicated* and **7 cached partitions** — the cache follows the DataFrame's partitions, not the number of files.
- Which operations immediately make that cache appear? Only those that include an **action**: counting the DataFrame (`partitioned_df.count()`) and querying and **showing** the result (`spark.sql("select count(*) from text").show()`). A transformation such as `distinct()`, or `spark.sql(...)` without `.show()`, only builds a plan.

### 3.3 — Logging without hidden costs

Python's `logging` module with levels lets me keep diagnostic messages in the code and choose how much is printed:

In [ ]:
import logging, sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Hello %s", "world")    # printed at INFO
logging.debug("Hello, take %d", 2)   # printed only when level=logging.DEBUG

**The trap — stealth CPU waste:** because of **lazy evaluation**, a log message that contains an action runs that action even when its level is filtered out. Arguments are evaluated before `logging` decides not to print:

```python
logging.debug("df has %d rows.", df.count())   # counts the whole DataFrame even at INFO level!
```

The fix is a flag that disables expensive debug actions explicitly:

In [ ]:
ENABLED = False

logger.info("No action here.")
if ENABLED:
    logger.info("df has %d rows.", df.count())   # only runs when explicitly enabled

The course also uses a small `timer` class (print elapsed seconds per step, then reset) to measure each step — the same tool behind the caching timings above.

**Exercise — practice logging.**

In [ ]:
# Log columns of text_df as debug message
logging.debug("text_df columns: %s", text_df.columns)

# Log whether table1 is cached as info message
logging.info("table1 is cached: %s", spark.catalog.isCached(tableName="table1"))

# Log first row of text_df as warning message
logging.warning("The first row of text_df:\n %s", text_df.first())

# Log selected columns of text_df as error message
logging.error("Selected columns: %s", text_df.select("id", "word"))

![At DEBUG level, Spark's own Py4J traffic is logged too, along with my message listing the text_df columns](./screenshots/ch3_ex6_logging_levels.png)
*At DEBUG level, Spark's own Py4J traffic is logged too, along with my message listing the text_df columns*

**Interpretation:** at DEBUG level the output fills with Spark's internal JVM communication, which is a reason to keep production logging at INFO and enable DEBUG only while investigating.

**Exercise — identifying log statements that trigger an action.** I left commented only the statements that run an action on `text_df` / `table1`:

In [ ]:
# Uncomment the 5 statements that do NOT trigger text_df
logging.debug("text_df columns: %s", text_df.columns)
logging.info("table1 is cached: %s", spark.catalog.isCached(tableName="table1"))
# logging.warning("The first row of text_df: %s", text_df.first())
logging.error("Selected columns: %s", text_df.select("id", "word"))
logging.info("Tables: %s", spark.sql("show tables").collect())
logging.debug("First row: %s", spark.sql("SELECT * FROM table1 limit 1"))
# logging.debug("Count: %s", spark.sql("SELECT COUNT(*) AS count FROM table1").collect())

**How to tell them apart:**
- **No action on the data:** `.columns` and `isCached()` only read metadata; `select()` and `spark.sql(...)` without an action are transformations that just build a plan.
- **Action:** `.first()` and `.collect()` force Spark to read the data — as do `.count()` and `.show()`.
- **The subtle case:** `spark.sql("show tables").collect()` *is* an action, but on the catalog, not on `text_df`, so it doesn't touch the table's data.

### 3.4 — Query plans

`EXPLAIN` (SQL) and `.explain()` (DataFrame) show the **physical plan** Spark will execute. Plans are read **bottom-up**: the scan of the source comes first, the final operation is at the top.

In [ ]:
df = spark.read.load('/temp/df.parquet')
df.createOrReplaceTempView('df')

spark.sql('EXPLAIN SELECT * FROM df').first()
df.explain()
spark.sql("SELECT * FROM df").explain()

# Words sorted by frequency — SQL and dot notation produce the same plan
spark.sql("SELECT word, COUNT(*) AS count FROM df GROUP BY word ORDER BY count DESC").explain()

from pyspark.sql.functions import desc
df.groupBy('word').count().sort(desc('count')).explain()

What to look for: `FileScan parquet` with `Location` (source file) and `ReadSchema` (columns actually read); `InMemoryRelation` / `InMemoryTableScan` when the data is cached; `HashAggregate` for grouping; and `Sort` for ordering. If a DataFrame was cached, the plan reads from memory instead of the file.

**Exercise — practice query plans.**

In [ ]:
# Run explain on text_df
text_df.explain()

# Run explain on "SELECT COUNT(*) AS count FROM table1"
spark.sql("SELECT COUNT(*) AS count FROM table1").explain()

# Run explain on "SELECT COUNT(DISTINCT word) AS words FROM table1"
spark.sql("SELECT COUNT(DISTINCT word) AS words FROM table1").explain()

![text_df: a Parquet scan with a filter on part = 2](./screenshots/ch3_ex8_explain_text_df.png)
*text_df: a Parquet scan with a filter on part = 2*

![COUNT(*): partial counts per partition, then one exchange to a single partition for the final count](./screenshots/ch3_ex8_explain_count.png)
*COUNT(*): partial counts per partition, then one exchange to a single partition for the final count*

![COUNT(DISTINCT word): an extra hash-partitioning shuffle on word before counting](./screenshots/ch3_ex8_explain_count_distinct.png)
*COUNT(DISTINCT word): an extra hash-partitioning shuffle on word before counting*

**Interpretation:**
- `text_df` is defined with a filter `part = 2`, and Spark pushes that filter down to the Parquet scan (`DataFilters`), reading only what it needs.
- `COUNT(*)` reads only the `part` column (`FileScan parquet [part#2]`) — Spark prunes unused columns automatically.
- `COUNT(DISTINCT word)` needs **two** exchanges: one `hashpartitioning(word, 200)` to bring identical words together, then a single partition for the final count. Distinct counts are noticeably more expensive than plain counts.

**Exercise — reading query plans 2.** Three DataFrames were inspected with `.explain()` only:

In [ ]:
part2_df.explain()
part3_df.explain()
part4_df.explain()

![part2_df's FileScan location](./screenshots/ch3_ex9_part2_source_file.png)
*part2_df's FileScan location*

- **Source of `part2_df`:** `sherlock_parts.parquet`.

![part3_df's plan: cached, and filtered on part = 4](./screenshots/ch3_ex9_part3_filter_bug.png)
*part3_df's plan: cached, and filtered on part = 4*

- **Bug in `part3_df`:** it was supposed to hold part 3, but the plan filters on **`part = 4`**. The plan reveals the bug without running the query on any data.

![part4_df's FileScan location](./screenshots/ch3_ex9_part4_source_file.png)
*part4_df's FileScan location*

![part4_df's ReadSchema](./screenshots/ch3_ex9_part4_read_schema.png)
*part4_df's ReadSchema*

- **Source of `part4_df`:** `sherlock.parquet`, and its **ReadSchema** is `struct<word:string,id:bigint>` — only two columns are read from the file.

Spark tags each column with an identifier (`word#0`, `id#1L`); these can be ignored when reading plans.

## Chapter 4 — Text Classification

### 4.1 — Extract, Transform, Select

The course frames feature preparation as **ETS: Extract, Transform, Select**. Built-in functions (`split`, `explode`, `length`) cover most transformations; when they don't, a **UDF** fills the gap.

In [ ]:
from pyspark.sql.functions import udf, length
from pyspark.sql.types import BooleanType, StringType, IntegerType, FloatType, ArrayType

df.where(length('sentence') == 0)   # find empty sentences with a built-in

# Boolean UDF
short_udf = udf(lambda x: True if not x or len(x) < 10 else False, BooleanType())
df.select(short_udf('textdata').alias("is short")).show(3)

# Array UDF: remove the last item of an array
in_udf = udf(lambda x: x[0:len(x)-1] if x and len(x) > 1 else [], ArrayType(StringType()))

**Sparse vectors** store only non-zero entries as `(size, [indices], [values])`. For example, `[1.0, 0.0, 0.0, 3.0]` becomes `(4, [0, 3], [1.0, 3.0])`. To handle them safely in a UDF: check `hasattr(x, "toArray")` (it is a vector) and `x.numNonzeros()` (it isn't empty).

**The return type must be cast properly:** `x.indices[0]` is a NumPy integer, and returning it from a UDF declared as `IntegerType()` fails with a `Py4JJavaError`. Wrapping it in `int(...)` fixes it.

**Exercise — creating UDFs.**

In [ ]:
from pyspark.sql.types import BooleanType, StringType

# Returns true if the value is a nonempty vector
nonempty_udf = udf(lambda x:
    True if (x and hasattr(x, "toArray") and x.numNonzeros())
    else False, BooleanType())

# Returns first element of the array as string
s_udf = udf(lambda x: str(x[0]) if (x and type(x) is list and len(x) > 0)
    else '', StringType())

**Exercise — practicing array columns.** Remove the words in `TRIVIAL_TOKENS` (a Python set) from the `in` and `out` columns.

In [ ]:
from pyspark.sql.functions import array_contains
from pyspark.sql.types import ArrayType, StringType

# Show the rows where doc contains the item '5'
df_before.where(array_contains('doc', '5')).show()

# UDF removes items in TRIVIAL_TOKENS from array
rm_trivial_udf = udf(lambda x:
                     list(set(x) - TRIVIAL_TOKENS) if x
                     else x,
                     ArrayType(StringType()))

# Remove trivial tokens from 'in' and 'out' columns of df2
df_after = df_before.withColumn('in', rm_trivial_udf('in'))\
                    .withColumn('out', rm_trivial_udf('out'))

# Show the rows of df_after where doc contains the item '5'
df_after.where(array_contains('doc','5')).show()

![Before and after: tokens like '5' are removed from in/out, while doc is untouched](./screenshots/ch4_ex2_remove_trivial_tokens.png)
*Before and after: tokens like '5' are removed from in/out, while doc is untouched*

**Interpretation:** `out` values that were only `[5]` become empty arrays, and `in` loses its trivial tokens. Using a set difference also changes the word order, which was acceptable here.

**Exercise — creating a UDF for vector data.** The `output` column holds sparse vectors; `first_udf` returns the index of the first non-zero entry, or `0.0` for anything that isn't a non-empty vector.

![Input: sparse vectors of size 12,847 with a single non-zero entry](./screenshots/ch4_ex3_vector_column_input.png)
*Input: sparse vectors of size 12,847 with a single non-zero entry*

In [ ]:
from pyspark.sql.types import FloatType

# Selects the first element of a vector column
first_udf = udf(lambda x:
            float(x.indices[0])
            if (x and hasattr(x, "toArray") and x.numNonzeros())
            else 0.0,
            FloatType())

# Apply first_udf to the output column
df.select(first_udf("output").alias("result")).show(5)

![The index of each vector's non-zero entry, as a float](./screenshots/ch4_ex3_first_udf_result.png)
*The index of each vector's non-zero entry, as a float*

**Interpretation:** the printed `(12847,[65],[1.0])` looks like a tuple, but it is a `SparseVector` object, so its parts are read through attributes — `x.size`, `x.indices`, `x.values` — not by position (`x[1]`). That's why the UDF uses `x.indices[0]`.

**Exercise — applying a UDF to vector data.**

In [ ]:
# Add label by applying the get_first_udf to output column
df_new = df.withColumn('label', get_first_udf('output'))

# Show the first five rows
df_new.show()

![A label column derived from each output vector](./screenshots/ch4_ex4_label_column.png)
*A label column derived from each output vector*

### 4.2 — Creating feature data with CountVectorizer

**`CountVectorizer`** is a feature extractor: its input is an array of strings and its output is a sparse vector of word counts over the learned vocabulary. It is fitted once, then used to transform data.

In [ ]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(inputCol='words', outputCol='features')
model = cv.fit(df)
result = model.transform(df)    # DataFrame[words: array<string>, features: vector]

**Exercise — transforming text to vector format.** The model expects a column `words` and creates `vec`. Columns `in` (sentence without its last word) and `out` (the last word) were vectorized by renaming around the model:

In [ ]:
# Transform df using model
result = model.transform(df.withColumnRenamed('in', 'words'))\
        .withColumnRenamed('words', 'in')\
        .withColumnRenamed('vec', 'invec')
result.drop('sentence').show(3, False)

# Add a column based on the out column called outvec
result = model.transform(result.withColumnRenamed('out', 'words'))\
        .withColumnRenamed('words', 'out')\
        .withColumnRenamed('vec', 'outvec')
result.select('invec', 'outvec').show(3, False)

![in vectorized into invec over a 126-word vocabulary](./screenshots/ch4_ex5_invec.png)
*in vectorized into invec over a 126-word vocabulary*

![out vectorized into outvec](./screenshots/ch4_ex5_outvec.png)
*out vectorized into outvec*

**Interpretation:** `[then, how, many, are]` becomes `(126,[3,18,28,30],[1.0,1.0,1.0,1.0])` — four vocabulary indices with count 1. Renaming in and out of `words` lets one fitted model encode both columns consistently.

### 4.3 — Text classification

The task: predict whether a sentence ends with a given word (e.g. a pronoun) from the preceding words. Steps: label the data, balance classes, split, train a logistic regression, then predict and evaluate.

In [ ]:
from pyspark.sql.functions import lit
from pyspark.ml.classification import LogisticRegression

df_true = df.where("endword in ('she', 'he', 'hers', 'his', 'her', 'him')").withColumn('label', lit(1))
df_false = df.where("endword not in ('she', 'he', 'hers', 'his', 'her', 'him')").withColumn('label', lit(0))
df_examples = df_true.union(df_false)

df_train, df_eval = df_examples.randomSplit((0.60, 0.40), 42)

logistic = LogisticRegression(maxIter=50, regParam=0.6, elasticNetParam=0.3)
model = logistic.fit(df_train)
print("Training iterations: ", model.summary.totalIterations)

predicted = model.transform(df_eval)        # adds prediction (double) and probability (vector of length 2)
model_stats = model.evaluate(df_eval)       # BinaryLogisticRegressionSummary
print("\nPerformance: %.2f" % model_stats.areaUnderROC)

In the course example, predicting pronoun endings with 2,873 positive + 2,873 negative examples reached a **test AUC of 0.87**, and predicting the word *"it"* reached 0.85 with far fewer examples.

**Exercise — labelling the data.** Positive examples end with *"him"*; I added an equal number of negatives so the classes are balanced.

In [ ]:
# Import the lit function
from pyspark.sql.functions import lit  # TODO: verificar — the screenshot shows the scaffold blank "pyspark.____"

# Select the rows where endword is 'him' and label 1
df_pos = df.where("endword = 'him'")\
           .withColumn('label', lit(1))

# Select the rows where endword is not 'him' and label 0
df_neg = df.where("endword <> 'him'")\
           .withColumn('label', lit(0))

# Union pos and neg in equal number
df_examples = df_pos.union(df_neg.limit(df_pos.count()))
print("Number of examples: ", df_examples.count())
df_examples.where("endword <> 'him'").sample(False, .1, 42).show(5)

![2,586 balanced examples; a sample of negatives with label 0](./screenshots/ch4_ex6_labeled_examples.png)
*2,586 balanced examples; a sample of negatives with label 0*

**Interpretation:** 2,586 examples = 1,293 positive + 1,293 negative. Limiting the negatives to `df_pos.count()` prevents the far more common non-*him* sentences from dominating the training data.

**Exercise — splitting the data (80/20).**

In [ ]:
# Split the examples into train and test, use 80/20 split
df_trainset, df_testset = df_examples.randomSplit((0.80, 0.20), 42)  # TODO: verificar — this line is cut off in the screenshot

# Print the number of training examples
print("Number training: ", df_trainset.count())

# Print the number of test examples
print("Number test: ", df_testset.count())

![2,120 training and 466 test examples](./screenshots/ch4_ex7_train_test_counts.png)
*2,120 training and 466 test examples*

**Interpretation:** 2,120 / 466 is about 82/18 — `randomSplit` uses the weights as probabilities, so the split is approximate rather than exact.

**Exercise — training the classifier.**

In [ ]:
# Import the logistic regression classifier
from pyspark.ml.classification import LogisticRegression  # TODO: verificar — the import line is cut off in the screenshot

# Instantiate logistic setting elasticnet to 0.0
logistic = LogisticRegression(maxIter=100, regParam=0.4, elasticNetParam=0.0)

# Train the logistic classifer on the trainset
df_fitted = logistic.fit(df_trainset)

# Print the number of training iterations
print("Training iterations: ", df_fitted.summary.totalIterations)

![The model converged in 17 iterations](./screenshots/ch4_ex8_training_iterations.png)
*The model converged in 17 iterations*

**Interpretation:** it converged after 17 iterations, well under the 100 allowed. `elasticNetParam=0.0` means pure L2 (ridge) regularization, with strength `regParam=0.4`.

**Exercise — evaluating the classifier.**

In [ ]:
# Score the model on test data
testSummary = df_fitted.evaluate(df_testset)

# Print the AUC metric
print("\ntest AUC: %.3f" % testSummary.areaUnderROC)

![Test AUC of 0.893](./screenshots/ch4_ex9_test_auc.png)
*Test AUC of 0.893*

**Interpretation:** an **AUC of 0.893** means the model ranks a random *him* sentence above a random non-*him* sentence about 89% of the time — slightly better than the 0.87 course benchmark for pronouns.

**Exercise — predicting test data.**

In [ ]:
# Apply the model to the test data
predictions = df_fitted.transform(df_testset).select(fields)

# Print incorrect if prediction does not match label
for x in predictions.take(8):
    print()
    if x.label != int(x.prediction):
        print("INCORRECT ==> ")
    for y in fields:
        print(y, ":", x[y])

![Predictions with class probabilities; one misclassified example flagged](./screenshots/ch4_ex10_predictions.png)
*Predictions with class probabilities; one misclassified example flagged*

**Interpretation:** the first two *him* sentences are predicted correctly with probabilities of 0.74 and 0.93. The third is wrong, but only narrowly (0.46 vs 0.54) — and the word *"him"* appears in the middle of that sentence, which plausibly confused a bag-of-words model that ignores word order.

## 🧠 Key Takeaways

- **Window functions replace self-joins:** `LEAD`, `LAG`, `ROW_NUMBER` and running `SUM`s return a value per row, and **`PARTITION BY` is essential** — without it, windows leak across groups (the row-7 bug).
- **SQL and dot notation are interchangeable**, but `.agg()` with a dictionary silently keeps only one aggregation per column; use functions with aliases, or SQL.
- **Windows power NLP features:** moving windows give context words and n-grams, and a windowed subquery with `ROW_NUMBER()` returns the top item per group.
- **Deterministic results need tie-breakers** in `ORDER BY`, and `DISTINCT` applies to the full row.
- **Cache the shared parent:** caching a reused upstream DataFrame sped up everything built on it (4.9 s vs 7.8 s).
- **Lazy evaluation has a cost in logging:** actions inside log arguments run even when the message isn't printed.
- **Query plans explain performance and catch bugs:** read them bottom-up for source files, pruned columns, pushed filters, shuffles (`Exchange`) and cached scans — and wrong filters like `part = 4`.
- **Spark ML pipeline basics:** UDFs for custom features, `CountVectorizer` for text, balanced labels, `randomSplit`, logistic regression and AUC.

## 💼 Application in a Data Analyst / Data Engineer Role

- **Event and time-series data:** time between events, sessionization, running totals and "previous value" comparisons all use the same `LEAD`/`LAG`/`SUM OVER` patterns used for the train schedule.
- **Top-N per group:** `ROW_NUMBER() OVER (PARTITION BY … ORDER BY …)` with `WHERE row = 1` is the standard way to keep the latest record per key or the top product per category — including deduplication in silver tables.
- **Performance and cost:** repartitioning on the grouping key, caching reused DataFrames and spotting shuffles in `explain()` reduce cluster time in Databricks jobs.
- **Production observability:** logging with levels, without hidden actions, keeps pipelines debuggable without slowing them down.
- **Feature engineering at scale:** text preparation and labelled datasets built with Spark feed ML models trained directly on lakehouse data.

## ✅ Skills Demonstrated

- Registering temp views and inspecting schemas with `DESCRIBE`, `SHOW COLUMNS` and `LIMIT 0`.
- Writing window functions (`LEAD`, `LAG`, `ROW_NUMBER`, running `SUM`) with `PARTITION BY` / `ORDER BY`, and translating them between SQL and the `Window` API.
- Using `UNIX_TIMESTAMP` / `unix_timestamp` for time differences.
- Text processing with `read.text`, `lower`, `regexp_replace`, `split`, `explode`, `length` and `monotonically_increasing_id`.
- Building context windows and n-gram counts with windowed subqueries, top-N per group, `DISTINCT` and deterministic ordering.
- Repartitioning by column and checking `getNumPartitions()`.
- Caching and persisting DataFrames and tables, timing the impact, and reading the Spark UI Storage tab.
- Logging with levels while avoiding hidden actions.
- Reading physical plans (`FileScan`, `ReadSchema`, `DataFilters`, `Exchange`, `HashAggregate`, `InMemoryTableScan`) to explain performance and find bugs.
- Writing UDFs for booleans, arrays and sparse vectors with correct return types.
- Building a text classifier with `CountVectorizer`, balanced labels, `randomSplit`, `LogisticRegression`, AUC evaluation and prediction review.